# Shapley-Owen Decomposition of McFadden R²

Exact Owen-value decomposition via the `elbersb/shapley` package. Partitions McFadden pseudo-R² among three groups:

- **Elite**: `mg_fsnub`, `mg_court` (20 km binary proximity dummies for Crown-grievance families and court officers)
- **Commons**: spec-specific continuous monastic variables (land, tithes, alms)
- **Controls**: `lpopC`, `lLStax_pc`, `mean_slope`

Variable values sum to the full-model McFadden R². Outputs LaTeX tables to `Output/Tables/`.
Run the setup cell first, then the decomposition cell.

In [1]:
pacman::p_load(
  sf, tidyverse, dplyr,
  lmtest, sandwich, stargazer,
  survival, jsonlite,
  car, AER, DoubleML, mlr3, mlr3learners, glmnet, conleyreg
)
pacman::p_load_gh("elbersb/shapley")

PROJECT_ROOT <- tryCatch(
  normalizePath(file.path(dirname(rstudioapi::getActiveDocumentContext()$path), "..")),
  error = function(e) {
    cwd <- normalizePath(getwd())
    if (basename(cwd) == "Code") dirname(cwd) else cwd
  }
)
setwd(PROJECT_ROOT)
pretty_dict <- fromJSON("Code/pretty_dict.json")

pdf <- read_sf(dsn = "Data/Processed/northParishFlows.shp")
pdf$mg_rnl_w <- pdf$mg_rebel_w + pdf$mg_neut_w + pdf$mg_loyal_w

## Standardize continuous variables (z-score)

In [2]:
continuous_vars <- c(
  "llandOwned", "llo_sk", "llo_arak",
  "lsmLand",   "lbigLand",
  "lsm_sk",    "lbg_sk",
  "lsm_arak",  "lbg_arak",
  "lotherLand", "lownLand",
  "loth_sk",    "lown_sk",
  "loth_arak",  "lown_arak",
  "ltitheOutT", "lti_sk", "lti_arak",
  "lalmsInTot", "lal_sk", "lal_arak",
                "lni_sk", "lni_arak",
  "mg_rnl_w",
  "lLStax_pc", "wet_1535", "wet_1536", "lpopC",
  "area", "mean_slope", "distScot"
)
for (v in continuous_vars) {
  pdf[[v]] <- scale(pdf[[v]], center = TRUE, scale = TRUE)[, 1]
}

## Variable sets

In [3]:
elite_vars <- c("mg_fsnub", "mg_court")

commons_specs <- list(
  total_raw  = c("llandOwned",                        "ltitheOutT", "lalmsInTot",  "smHouse", "bigHouse", "friary"),
  total_sk   = c("llo_sk",                            "lti_sk",     "lal_sk",      "smHouse", "bigHouse", "friary"),
  total_arak = c("llo_arak",                          "lti_arak",   "lal_arak",  "lni_arak",
                 "smHouse", "bigHouse", "friary"),
  split_raw  = c("lsmLand",  "lbigLand",              "ltitheOutT", "lalmsInTot",  "smHouse", "bigHouse", "friary"),
  split_sk   = c("lsm_sk",   "lbg_sk",                "lti_sk",     "lal_sk",      "smHouse", "bigHouse", "friary"),
  split_arak = c("lsm_arak", "lbg_arak",              "lti_arak",   "lal_arak",  "lni_arak",
                 "smHouse", "bigHouse", "friary"),
  ownOther_raw  = c("lotherLand", "lownLand",          "ltitheOutT", "lalmsInTot",  "smHouse", "bigHouse", "friary"),
  ownOther_sk   = c("loth_sk",    "lown_sk",           "lti_sk",     "lal_sk",      "smHouse", "bigHouse", "friary"),
  ownOther_arak = c("loth_arak",  "lown_arak",         "lti_arak",   "lal_arak",  "lni_arak",
                    "smHouse", "bigHouse", "friary")
)

commons_spec_suffix <- list(
  total_raw     = "_total_raw",    total_sk     = "_total_sk",   total_arak     = "_total_arak",
  split_raw     = "_split_raw",    split_sk     = "_split_sk",   split_arak     = "_split_arak",
  ownOther_raw  = "_ownOther_raw", ownOther_sk  = "_ownOther_sk",
  ownOther_arak = "_ownOther_arak"
)

controls <- c(
  "lLStax_pc", "wet_1535", "wet_1536", "lpopC",
  "uplands", "lowlands", "area", "mean_slope", "distScot"
)

outcomes <- list(
  muster  = list(dep = "muster",  family = binomial(link = "logit")),
  primary = list(dep = "primary", family = binomial(link = "logit")),
  seats   = list(dep = "seats",   family = poisson())
)

df <- as.data.frame(sf::st_drop_geometry(pdf))

## Shapley-Owen decomposition of McFadden R²

Exact Owen-value decomposition via the `elbersb/shapley` package, with a
McFadden R² value function. Restricted to the most predictive variables
across the broader specifications, partitioned into:

- **elite**: `mg_fsnub`, `mg_court` (Crown grievance channels)
- **commons**: spec's continuous monastic variables (land, tithes, alms)
- **controls**: `lpopC`, `lLStax_pc`, `mean_slope`

Variable values sum to the full-model McFadden R²; group values are the
sum of variable values within each group.

In [4]:
owen_decomposition <- function(data, outcome, groups,
                               family = binomial(link = "logit")) {
  null_ll <- as.numeric(logLik(suppressWarnings(
    glm(as.formula(paste(outcome, "~ 1")), data = data, family = family)
  )))
  value_fun <- function(factors) {
    if (length(factors) == 0) return(0)
    f <- as.formula(paste(outcome, "~", paste(factors, collapse = " + ")))
    m <- suppressWarnings(glm(f, data = data, family = family))
    1 - as.numeric(logLik(m)) / null_ll
  }
  owen_df  <- shapley::owen(value_fun, groups, silent = TRUE)
  var_vals <- setNames(owen_df$value, owen_df$factor)
  list(
    variable = var_vals,
    group    = sapply(groups, function(g) sum(var_vals[g])),
    total_r2 = value_fun(unlist(groups, use.names = FALSE))
  )
}

# Groups: commons vars are the spec's continuous vars (binary presence dummies excluded)
commons_binary_controls <- c("smHouse", "bigHouse", "friary")
so_groups_by_spec <- lapply(commons_specs, function(cvars) list(
  elite    = elite_vars,
  commons  = cvars[!cvars %in% commons_binary_controls],
  controls = c("lpopC", "lLStax_pc", "mean_slope")
))

cat("\n========== SHAPLEY-OWEN DECOMPOSITION ==========\n")
cat("(Exact Owen values via `shapley` package; may take several minutes)\n")

so_print <- function(tag, d, groups) {
  cat(sprintf("\n--- %s (total R² = %.4f) ---\n", tag, d$total_r2))
  for (gname in names(groups)) {
    cat(sprintf("  [%s]  group = %.4f\n", gname, d$group[gname]))
    for (v in groups[[gname]]) {
      cat(sprintf("    %-25s %.4f\n", v, d$variable[v]))
    }
  }
}

so_results <- list()
for (spec_name in names(so_groups_by_spec)) {
  g <- so_groups_by_spec[[spec_name]]
  cat(sprintf("\nDecompositions [%s]...\n", toupper(spec_name)))
  so_results[[spec_name]] <- list(
    muster  = owen_decomposition(df, "muster",  g, family = binomial(link = "logit")),
    primary = owen_decomposition(df, "primary", g, family = binomial(link = "logit")),
    seats   = owen_decomposition(df, "seats",   g, family = poisson())
  )
  so_print("MUSTER",  so_results[[spec_name]]$muster,  g)
  so_print("PRIMARY", so_results[[spec_name]]$primary, g)
  so_print("SEATS",   so_results[[spec_name]]$seats,   g)
}


========== SHAPLEY-OWEN DECOMPOSITION ==========
(Exact Owen values via `shapley` package; may take several minutes)

Decompositions [TOTAL_RAW]...

--- MUSTER (total R² = 0.1971) ---
  [elite]  group = 0.0024
    mg_fsnub                  0.0020
    mg_court                  0.0004
  [commons]  group = 0.0262
    llandOwned                0.0093
    ltitheOutT                0.0027
    lalmsInTot                0.0141
  [controls]  group = 0.1686
    lpopC                     0.0535
    lLStax_pc                 0.0475
    mean_slope                0.0675

--- PRIMARY (total R² = 0.2347) ---
  [elite]  group = 0.0010
    mg_fsnub                  0.0007
    mg_court                  0.0003
  [commons]  group = 0.0359
    llandOwned                0.0197
    ltitheOutT                0.0019
    lalmsInTot                0.0143
  [controls]  group = 0.1978
    lpopC                     0.0682
    lLStax_pc                 0.0561
    mean_slope                0.0735

--- SEATS (total R²

## Write LaTeX tables

In [5]:
group_titles <- c(elite = "Elite", commons = "Commons", controls = "Controls")

write_owen_table <- function(decompositions, groups, labels, caption, label, out_path) {
  outcome_cols <- toupper(names(decompositions))
  n_out <- length(decompositions)
  lines <- c(
    "\\begin{table}[H]",
    "\\centering",
    paste0("\\caption{", caption, "}"),
    paste0("\\label{", label, "}"),
    paste0("\\begin{tabular}{l", strrep("r", n_out), "}"),
    "\\hline\\hline",
    paste0("Variable & ", paste(outcome_cols, collapse = " & "), " \\\\"),
    "\\hline"
  )
  for (gname in names(groups)) {
    gvals <- sapply(decompositions, function(d) d$group[gname])
    lines <- c(lines, paste0(
      "\\textbf{", group_titles[gname], "} & ",
      paste(sprintf("\\textbf{%.4f}", gvals), collapse = " & "), " \\\\"
    ))
    for (v in groups[[gname]]) {
      vvals <- sapply(decompositions, function(d) d$variable[v])
      lines <- c(lines, paste0(
        "\\quad ", labels[v], " & ",
        paste(sprintf("%.4f", vvals), collapse = " & "), " \\\\"
      ))
    }
  }
  tvals <- sapply(decompositions, function(d) d$total_r2)
  lines <- c(
    lines, "\\hline",
    paste0("Total McFadden $R^2$ & ", paste(sprintf("%.4f", tvals), collapse = " & "), " \\\\"),
    "\\hline\\hline",
    "\\end{tabular}",
    "\\end{table}"
  )
  writeLines(lines, out_path)
}

for (spec_name in names(so_results)) {
  sfx <- commons_spec_suffix[[spec_name]]
  g   <- so_groups_by_spec[[spec_name]]
  write_owen_table(
    so_results[[spec_name]], g,
    unlist(pretty_dict[unlist(g)]),
    caption  = paste0("Shapley--Owen Decomposition of McFadden $R^2$ [", spec_name, "]"),
    label    = paste0("tab:shapley_owen", sfx),
    out_path = paste0("Output/Tables/elite_vs_commons_shapley_owen", sfx, ".tex")
  )
}

cat("\nShapley-Owen tables written to Output/Tables/",
    "elite_vs_commons_shapley_owen_{...}.tex\n", sep = "")


Shapley-Owen tables written to Output/Tables/elite_vs_commons_shapley_owen_{...}.tex
